# Lab 8: Anomaly Detection (Outlier Detection)

## 1. Introduction
Anomaly detection is the process of identifying data points, events, and observations that deviate significantly from a dataset's normal behavior. Outliers can often indicate critical incidents, such as technical glitches, or potential opportunities like a change in consumer behavior.

### Working Assumption:
There are considerably more "normal" observations than "abnormal" observations (outliers/anomalies) in the data.

### General Steps:
1. **Build a profile** of the "normal" behavior (patterns or summary statistics).
2. **Use the profile** to detect anomalies (observations that differ significantly from the profile).

### Approaches covered in Chapter 8:
1. **Graphical**: Box plots, Scatter plots.
2. **Statistical**: Z-score, Likelihood-based.
3. **Distance-based**: K-Nearest Neighbors (KNN), Local Outlier Factor (LOF).
4. **Clustering-based**: DBSCAN, Small cluster detection.
5. **Model-based**: Isolation Forest.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from scipy import stats

# Set plot style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Generating Synthetic Data
We will create a dataset with a dense "normal" cluster and some scattered outliers to test our algorithms.

In [ ]:
np.random.seed(42)

# Generate 200 normal points
X_normal, _ = make_blobs(n_samples=200, centers=1, cluster_std=1.0, center_box=(-10.0, 10.0))

# Generate 20 random outliers
X_outliers = np.random.uniform(low=-10, high=10, size=(20, 2))

# Combine data
X = np.r_[X_normal, X_outliers]
df = pd.DataFrame(X, columns=['Feature 1', 'Feature 2'])

plt.scatter(df['Feature 1'], df['Feature 2'], c='blue', edgecolors='k', alpha=0.7)
plt.title("Synthetic Dataset for Anomaly Detection")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## 3. Graphical Approaches
Graphical methods like Boxplots (1-D) and Scatter plots (2-D) provide visual intuition but are subjective and harder to scale.

### Interquartile Range (IQR) and Boxplots
Outliers are often defined as points falling below $Q1 - 1.5 \times IQR$ or above $Q3 + 1.5 \times IQR$.

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(y=df['Feature 1'], color='skyblue')
plt.title("Boxplot of Feature 1")

plt.subplot(1, 2, 2)
sns.boxplot(y=df['Feature 2'], color='salmon')
plt.title("Boxplot of Feature 2")
plt.show()

## 4. Statistical Approaches
### Z-Score (Parametric)
Assuming a normal distribution, the Z-score measures how many standard deviations a point is from the mean.
$$Z = \frac{x - \mu}{\sigma}$$
Typically, $|Z| > 3$ is considered an outlier.

In [ ]:
z_scores = np.abs(stats.zscore(df))
threshold = 3
outliers_mask = (z_scores > threshold).any(axis=1)

df_z = df.copy()
df_z['is_outlier'] = outliers_mask

sns.scatterplot(data=df_z, x='Feature 1', y='Feature 2', hue='is_outlier', palette={True: 'red', False: 'blue'})
plt.title("Outliers Detected by Z-Score (Threshold=3)")
plt.show()

## 5. Distance-based Approaches
### K-Nearest Neighbors (KNN)
A point is an outlier if its distance to its $k$-th nearest neighbor is large, or if it has fewer than $p$ neighbors within distance $D$.

In [ ]:
k = 5
knn = NearestNeighbors(n_neighbors=k)
knn.fit(X)
distances, indices = knn.kneighbors(X)

# Calculate average distance to k neighbors
avg_distances = distances.mean(axis=1)
threshold_knn = np.percentile(avg_distances, 95) # Top 5% defined as outliers

df_knn = df.copy()
df_knn['is_outlier'] = avg_distances > threshold_knn

sns.scatterplot(data=df_knn, x='Feature 1', y='Feature 2', hue='is_outlier', palette={True: 'red', False: 'blue'})
plt.title("Outliers Detected by KNN Distance (Top 5%)")
plt.show()

## 6. Density-based Approaches
### Local Outlier Factor (LOF)
LOF identifies outliers by comparing the local density of a point with the local densities of its neighbors.
$$LOF(P) = \frac{\text{Average Density of Neighbors}}{\text{Density of Sample P}}$$
Points with LOF much greater than 1 are outliers.

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
y_pred = lof.fit_predict(X)

df_lof = df.copy()
df_lof['is_outlier'] = y_pred == -1

sns.scatterplot(data=df_lof, x='Feature 1', y='Feature 2', hue='is_outlier', palette={True: 'red', False: 'blue'})
plt.title("Outliers Detected by Local Outlier Factor (LOF)")
plt.show()

## 7. Model-based Approaches
### Isolation Forest
Isolation Forest isolates observations by randomly selecting a feature and then randomly selecting a split value. Since outliers are "few and different," they are isolated much faster (shorter path length in the tree) than normal points.

In [ ]:
iso_forest = IsolationForest(contamination=0.1, random_state=42)
y_pred_iso = iso_forest.fit_predict(X)

df_iso = df.copy()
df_iso['is_outlier'] = y_pred_iso == -1

sns.scatterplot(data=df_iso, x='Feature 1', y='Feature 2', hue='is_outlier', palette={True: 'red', False: 'blue'})
plt.title("Outliers Detected by Isolation Forest")
plt.show()

## 8. Clustering-based Approaches
### DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
DBSCAN groups points that are close to each other. Points that do not belong to any cluster (often in low-density regions) are labeled as noise (-1).

In [ ]:
dbscan = DBSCAN(eps=1.0, min_samples=5)
clusters = dbscan.fit_predict(X)

df_dbscan = df.copy()
df_dbscan['is_outlier'] = clusters == -1

sns.scatterplot(data=df_dbscan, x='Feature 1', y='Feature 2', hue='is_outlier', palette={True: 'red', False: 'blue'})
plt.title("Outliers (Noise) Detected by DBSCAN")
plt.show()

## 9. Issues in Anomaly Detection
According to Chapter 8, several factors can complicate anomaly detection:

- **Number of Attributes**: An object might be anomalous in a high-dimensional space even if each individual attribute looks normal (Curse of Dimensionality).
- **Global vs. Local Perspective**: A point might be an outlier relative to its local neighbors but appear normal in a global context.
- **Degree of Anomaly**: Some points are "more" anomalous than others; it's often not binary.
- **One at a Time vs. Many at Once**: Some outliers might mask each other or appear normal when processed together.
- **Understandability (Interpretability)**: Users often need to know *why* a point was flagged (e.g., fraud detection).

## 10. Base Rate Fallacy
The **Base Rate Fallacy** is the tendency to ignore general information (base rates) in favor of specific information. In detection systems (like disease testing or fraud detection), if the event is very rare, even an accurate test can have a high number of false positives.

### Bayes' Theorem:
$$P(\text{Disease} | \text{Pos}) = \frac{P(\text{Pos} | \text{Disease}) \times P(\text{Disease})}{P(\text{Pos})}$$

### Example Calculation (from Slides):
- **Prevalence (Base Rate)**: 1 in 1,000 (0.1%)
- **True Positive Rate (Sensitivity)**: 99%
- **False Positive Rate**: 1%

In [ ]:
prevalence = 0.001
sensitivity = 0.99
fpr = 0.01

# Total probability of Testing Positive
p_pos = (sensitivity * prevalence) + (fpr * (1 - prevalence))

# Posterior probability: P(Disease | Positive Test)
p_disease_given_pos = (sensitivity * prevalence) / p_pos

print(f"Base Rate (Prevalence): {prevalence*100}%")
print(f"Probability of actually having the disease given a positive test: {p_disease_given_pos:.4f} ({p_disease_given_pos*100:.2f}%)")

## 11. Exercises
1. **Parameter Tuning**: Re-run the LOF and Isolation Forest code with different `contamination` values (e.g., 0.01, 0.05, 0.2). How do the results change?
2. **DBSCAN Challenge**: Change the `eps` and `min_samples` parameters in the DBSCAN section. Can you find a setting that identifies all 20 manually generated outliers without flagging normal points?
3. **Real-world Scenario**: Given a 95% accurate security alarm and a crime rate of 0.01%, calculate the probability that a crime is actually occurring if the alarm goes off.